# Silver - Lakehouse Medalhão de RH (Databricks)

Este notebook é a versão em PySpark, para o **Databricks**, do mini Lakehouse em pandas construído na seção "Modelagem Dimensional na Prática" do notebook `demonstracao-lakehouse-medalhao.ipynb` (pasta `aula-big-data/`). A lógica de transformação é idêntica à versão para o Fabric (`scripts/fabric/silver.ipynb`) — a única diferença real é como as tabelas são endereçadas: aqui usamos o namespace de três níveis do **Unity Catalog** (`catalogo.schema.tabela`) em vez de nomes de tabela dentro de um Lakehouse.

> ⚠️ **Pré-requisito:** este notebook não roda no Colab nem localmente — ele espera um workspace do Databricks com **Unity Catalog** habilitado, um catálogo com os schemas `bronze` e `silver` já criados, e a tabela `bronze.departamentos`/`bronze.cargos`/`bronze.funcionarios`/`bronze.eventos` já populada pela ingestão descrita no passo a passo da demonstração.

In [ ]:
dbutils.widgets.text('catalogo', 'meu_catalogo')
catalogo = dbutils.widgets.get('catalogo')

from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Dimensões Tipo 1: `departamentos` e `cargos`

Equivalente Spark de `etl_silver_departamentos` / `etl_silver_dimensao_tipo1` (pandas): mantém apenas o registro mais recente de cada chave, usando uma janela ordenada por `data_extracao` em vez de `sort_values().drop_duplicates()`.

In [ ]:
def silver_dimensao_tipo1(nome_tabela, coluna_chave):
    df_bronze = spark.table(f'{catalogo}.bronze.{nome_tabela}')

    # janela por chave, ordenada da extração mais recente para a mais antiga
    janela = Window.partitionBy(coluna_chave).orderBy(F.col('data_extracao').desc())

    # mantém só a linha mais recente de cada chave (SCD Tipo 1), descartando os metadados de auditoria do Bronze
    df_silver = (
        df_bronze
        .withColumn('_rn', F.row_number().over(janela))
        .filter(F.col('_rn') == 1)
        .drop('_rn', '_arquivo_origem', '_timestamp_ingestao')
    )

    df_silver.write.format('delta').mode('overwrite').saveAsTable(f'{catalogo}.silver.{nome_tabela}')
    print(f'[SILVER] "{nome_tabela}" atualizada com {df_silver.count()} registros (SCD Tipo 1).')
    return df_silver


df_departamentos = silver_dimensao_tipo1('departamentos', 'departamento_id')
df_cargos = silver_dimensao_tipo1('cargos', 'cargo_id')

## Dimensão Tipo 2: `dim_funcionario`

Equivalente Spark de `etl_silver_dim_funcionario_scd2` (pandas): `shift()`/`groupby` viram `lag()`/`Window`, e o *forward-fill* de `data_inicio_vigencia` vira `F.last(..., ignorenulls=True)` sobre uma janela `unboundedPreceding` até a linha atual.

In [ ]:
def silver_dim_funcionario_scd2():
    df = spark.table(f'{catalogo}.bronze.funcionarios').withColumn('data_extracao', F.to_date('data_extracao'))

    janela_funcionario = Window.partitionBy('funcionario_id').orderBy('data_extracao')

    # compara cada extração com a extração anterior do mesmo funcionário
    df = (
        df
        .withColumn('_dep_anterior', F.lag('departamento_id').over(janela_funcionario))
        .withColumn('_cargo_anterior', F.lag('cargo_id').over(janela_funcionario))
        .withColumn('_status_anterior', F.lag('status').over(janela_funcionario))
    )

    # nova versão: primeira extração do funcionário, ou alguma coluna rastreada mudou
    df = df.withColumn(
        '_nova_versao',
        F.col('_dep_anterior').isNull()
        | (F.col('departamento_id') != F.col('_dep_anterior'))
        | (F.col('cargo_id') != F.col('_cargo_anterior'))
        | (F.col('status') != F.col('_status_anterior')),
    )

    # início de vigência = primeira extração em que a combinação atual apareceu (forward-fill via janela)
    janela_ffill = Window.partitionBy('funcionario_id').orderBy('data_extracao').rowsBetween(Window.unboundedPreceding, 0)
    df = df.withColumn(
        'data_inicio_vigencia',
        F.last(F.when(F.col('_nova_versao'), F.col('data_extracao')), ignorenulls=True).over(janela_ffill),
    )

    # mantém só as linhas que de fato abrem uma nova versão
    df_versoes = df.filter(F.col('_nova_versao')).drop(
        '_dep_anterior', '_cargo_anterior', '_status_anterior', '_nova_versao',
        'data_extracao', '_arquivo_origem', '_timestamp_ingestao',
    )

    # fim de vigência = início da próxima versão do mesmo funcionário menos um dia; nulo = versão ainda vigente
    janela_versoes = Window.partitionBy('funcionario_id').orderBy('data_inicio_vigencia')
    df_versoes = (
        df_versoes
        .withColumn('data_fim_vigencia', F.date_sub(F.lead('data_inicio_vigencia').over(janela_versoes), 1))
        .withColumn('is_current', F.col('data_fim_vigencia').isNull())
    )

    # chave substituta (surrogate key) sequencial, uma por versão
    janela_sk = Window.orderBy('funcionario_id', 'data_inicio_vigencia')
    df_versoes = df_versoes.withColumn('sk_funcionario', F.row_number().over(janela_sk))

    df_versoes.write.format('delta').mode('overwrite').saveAsTable(f'{catalogo}.silver.dim_funcionario')
    print(f'[SILVER] dim_funcionario com {df_versoes.count()} versões (SCD Tipo 2).')
    return df_versoes


df_dim_funcionario = silver_dim_funcionario_scd2()

> 💡 **Dica:** `row_number()` sobre uma janela sem `partitionBy` (como em `janela_sk`) força o Spark a mover todos os dados para uma única partição — aceitável para uma dimensão pequena como esta, mas não é a estratégia recomendada para gerar chaves substitutas em tabelas grandes. Em produção, prefira uma chave determinística (por exemplo, um hash da chave natural com a data de início de vigência).

## Tabela Fato: `eventos` (Silver)

Equivalente Spark de `etl_silver_eventos` (pandas): `eventos` já chega em formato de fato de transação, então a Silver só tipa a data e remove os metadados de auditoria — sem deduplicação.

In [ ]:
def silver_eventos():
    df_bronze = spark.table(f'{catalogo}.bronze.eventos')

    df_silver = (
        df_bronze
        .withColumn('data_evento', F.to_date('data_evento'))
        .drop('_arquivo_origem', '_timestamp_ingestao')
    )

    df_silver.write.format('delta').mode('overwrite').saveAsTable(f'{catalogo}.silver.eventos')
    print(f'[SILVER] {df_silver.count()} eventos (sem deduplicação — cada evento é único por natureza).')
    return df_silver


df_eventos = silver_eventos()

Com `departamentos`, `cargos`, `dim_funcionario` e `eventos` publicados em `{catalogo}.silver`, o próximo passo é o notebook `gold.ipynb` desta mesma pasta, que monta `estrutura_organizacional` e a tabela fato `fato_eventos_rh`.